In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/sml project/datasets/Final_Augmented_dataset_Diseases_and_Symptoms.csv')

### Data Cleaning and Preprocessing for Model Training

Given that some disease classes might have very few instances, we'll first analyze the distribution of the 'diseases' column. Classes with very low counts can cause issues during model training (e.g., a model might not learn patterns for them or they might not appear in both training and test sets). We'll also check for any missing values and then prepare the dataset by separating features and the target variable, and encoding the target.

In [ ]:
# 1. Analyze 'diseases' column distribution
disease_counts = df['diseases'].value_counts()
display(disease_counts.head(10))
display(disease_counts.tail(10))

# Identify diseases with low frequency (e.g., less than 5 samples)
rare_diseases = disease_counts[disease_counts < 80].index
print(f"\nNumber of rare diseases (less than 5 samples): {len(rare_diseases)}")
print("Rare diseases to be removed or grouped:", list(rare_diseases))

,count
diseases,
cystitis,1219
vulvodynia,1218
nose disorder,1218
complex regional pain syndrome,1217
spondylosis,1216
esophagitis,1215
conjunctivitis due to allergy,1215
hypoglycemia,1215
peripheral nerve disorder,1215


,count
diseases,
heat stroke,1
chronic ulcer,1
open wound of the knee,1
open wound of the chest,1
open wound due to trauma,1
thalassemia,1
huntington disease,1
typhoid fever,1
kaposi sarcoma,1



Number of rare diseases (less than 5 samples): 298
Rare diseases to be removed or grouped: ['somatization disorder', 'fracture of the patella', 'toxic multinodular goiter', 'lewy body dementia', 'esophageal cancer', 'vaginismus', 'chorioretinitis', 'menopause', 'goiter', 'epilepsy', 'fracture of the jaw', 'parasitic disease', 'hypernatremia', 'hypothyroidism', 'adhesive capsulitis of the shoulder', 'ascending cholangitis', 'pityriasis rosea', 'atonic bladder', 'aplastic anemia', 'central retinal artery or vein occlusion', 'atrial fibrillation', 'ovarian torsion', 'poisoning due to analgesics', 'lice', 'fracture of the skull', 'protein deficiency', 'dislocation of the shoulder', 'injury to the abdomen', 'crushing injury', 'drug abuse (barbiturates)', 'septic arthritis', 'salivary gland disorder', 'nonalcoholic liver disease (nash)', 'hydatidiform mole', 'endophthalmitis', 'testicular disorder', 'amyotrophic lateral sclerosis (als)', 'hematoma', 'phimosis', 'lichen simplex', 'guillain b

From the value counts, we can identify diseases that appear very infrequently. For a robust model, it's often best to either remove these rare classes or group them into an 'Other' category if they are too numerous. For now, we will remove rows corresponding to diseases with less than 5 samples to ensure a minimum representation for each class.

In [ ]:
# Strip whitespace and convert to lowercase for both
df['diseases'] = df['diseases'].str.strip().str.lower()
rare_diseases = [d.strip().lower() for d in rare_diseases]

# Now run the filter
df_cleaned = df[~df['diseases'].isin(rare_diseases)]

print(f"Original DataFrame shape: {df.shape}")
print(f"Cleaned DataFrame shape after removing rare diseases: {df_cleaned.shape}")

unique_count = df['diseases'].nunique()
print(f"Number of unique diseases: {unique_count}")

unique_count = df_cleaned['diseases'].nunique()
print(f"Number of unique diseases: {unique_count}")

# Verify the new distribution
display(df_cleaned['diseases'].value_counts().tail(10))

Original DataFrame shape: (246945, 378)
Cleaned DataFrame shape after removing rare diseases: (239444, 378)
Number of unique diseases: 773
Number of unique diseases: 475


,count
diseases,
achalasia,84
bone cancer,83
endocarditis,83
gynecomastia,83
dyshidrosis,83
extrapyramidal effect of drugs,83
spina bifida,82
spermatocele,81
kidney cancer,80


Next, we will check for any missing values across all columns in the cleaned DataFrame. While `df.describe()` on the original DataFrame showed no missing values for numerical columns, it's a good practice to re-verify for the entire dataset after any preprocessing steps.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Separate features (X) and target (y)
X = df_cleaned.drop('diseases', axis=1)
y_disease_names = df_cleaned['diseases']

# Encode the target variable
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_disease_names)

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")
print("Encoded target classes:", label_encoder.classes_[:5], '...') # Display first 5 classes
print("Example original disease names:", y_disease_names[:5].tolist())
print("Example encoded labels:", y[:5].tolist())

Shape of features (X): (239444, 377)
Shape of target (y): (239444,)
Encoded target classes: ['abdominal aortic aneurysm' 'abdominal hernia' 'abscess of nose'
 'abscess of the pharynx' 'achalasia'] ...
Example original disease names: ['panic disorder', 'panic disorder', 'panic disorder', 'panic disorder', 'panic disorder']
Example encoded labels: [319, 319, 319, 319, 319]


### Optimize Feature Data Types for Memory Efficiency

Since the symptom columns are binary (0s and 1s), they can be stored using a much smaller data type (`int8` or `bool`) instead of the default `int64` or `float64`. This will significantly reduce the memory footprint of our feature matrix `X`.

In [ ]:
# Convert feature columns (symptoms) to int8 to reduce memory usage
# Assuming symptom columns are binary (0 or 1)

initial_memory_usage_X = X.memory_usage(deep=True).sum() / (1024**2)
print(f"Initial memory usage of X: {initial_memory_usage_X:.2f} MB")

for col in X.columns:
    if X[col].dtype == 'int64': # Only target integer columns that might be binary
        if X[col].isin([0, 1]).all(): # Check if all values are 0 or 1
            X[col] = X[col].astype('int8')

optimized_memory_usage_X = X.memory_usage(deep=True).sum() / (1024**2)
print(f"Optimized memory usage of X: {optimized_memory_usage_X:.2f} MB")
print(f"Memory reduced by: {initial_memory_usage_X - optimized_memory_usage_X:.2f} MB")

# Verify a sample of column dtypes
print("\nSample dtypes after optimization:")
display(X.head(2))

Initial memory usage of X: 711.82 MB
Optimized memory usage of X: 90.62 MB
Memory reduced by: 621.19 MB

Sample dtypes after optimization:


,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,palpitations,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,1,0,1,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import gc

# Explicitly delete the original DataFrame to free up memory if it still exists
if 'df' in locals() or 'df' in globals():
    del df
gc.collect()

print("Original DataFrame 'df' deleted and memory freed (if it existed).")

Original DataFrame 'df' deleted and memory freed (if it existed).


### Model Training: RandomForestClassifier

We will train a RandomForestClassifier, a popular ensemble learning method known for its good performance and ability to handle high-dimensional data. This model will learn to predict diseases based on the provided symptoms.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (191555, 377)
Shape of X_test: (47889, 377)
Shape of y_train: (191555,)
Shape of y_test: (47889,)


In [15]:
from sklearn.linear_model import LogisticRegression

# This will be MUCH faster for 247k rows
lr = LogisticRegression(solver='saga', max_iter=1000, n_jobs=-1)
lr.fit(X_train, y_train)

KeyboardInterrupt: 